# Evaluación del agente LLM

Compara el LLM por API (Groq) con Qwen2.5 local en 4 bits.

Configuración: acelerador **GPU T4**, Internet **On** y, en *Add-ons > Secrets*, el secreto `GROQ_API_KEY`.

In [ ]:
!git clone -q https://github.com/jazminsoto1234/Lego.git /kaggle/working/Lego
%cd /kaggle/working/Lego/agente
!pip -q install -U bitsandbytes accelerate "transformers>=4.45" pydantic

In [ ]:
!python test_agente.py

## LLM por API

In [ ]:
from llms import LLMAPI
api = LLMAPI('groq')
print(api.modelos())
print(api('Responde solo OK', 'hola', max_tokens=50))

## LLM local (Qwen2.5-7B en 4 bits; si no entra, 3B)

In [ ]:
from llms import cargar_local
local = cargar_local()
print(local.nombre, local('Responde solo OK', 'hola', max_tokens=10))

## Prueba rápida

In [ ]:
import json
from agente import interpretar, resumir
reg = json.load(open('registros_sinteticos/sesion_01.json', encoding='utf-8'))
for llm in (api, local):
    print('====', llm.nombre)
    print(interpretar('ponle lentes y que se vea más serio', llm))
    print(interpretar('que se parezca a mi profe de física', llm))
    print(resumir(reg, llm))
    print(reg['resumen']['origen'], reg['resumen']['intentos'])

## Evaluación completa (sesiones sintéticas y reales)

In [ ]:
import sys, evaluar
construidos = {'groq': api, 'local': local, 'ninguno': None}
evaluar.construir_llm = lambda n: construidos[n]  # reutiliza los modelos ya cargados
sys.argv = ['evaluar.py', '--llms', 'groq,local,ninguno']
evaluar.main()
print(open('resultados/tabla_agente.md', encoding='utf-8').read())